# Notebook 06: ADAPT-VQE vs UCCSD-VQE
## Adaptive vs Fixed Ansatz for Alkenes and Alkynes on NISQ Hardware

**Central question:** For molecules with strong π-correlation (alkynes), does
ADAPT-VQE's adaptive circuit growth outperform fixed UCCSD-VQE in both
accuracy *and* circuit depth — the two key metrics for NISQ viability?

**Reference:** Grimsley et al., *Nat. Commun.* **10**, 3007 (2019)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/quantum-alkene-alkyne-pyscf/blob/main/notebooks/06_adapt_vqe_comparison.ipynb)

In [ ]:
# CELL 1: Install
!pip install -q pyscf openfermion openfermionpyscf pennylane qiskit qiskit-aer scipy 2>&1 | tail -5
print('Ready.')

In [ ]:
# CELL 2: Imports
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
from scipy.optimize import minimize

from openfermion.chem import MolecularData
from openfermion import get_fermion_operator, jordan_wigner
from openfermion.utils import count_qubits
from openfermion.transforms import freeze_orbitals
from openfermionpyscf import run_pyscf
import pennylane as qml
from pennylane import qchem
print('Imports OK.')

In [ ]:
# CELL 3: Helper — OpenFermion QubitOperator → PennyLane Hamiltonian
def of_to_pl(qubit_op):
    coeffs, ops = [], []
    for term, c in qubit_op.terms.items():
        coeffs.append(np.real(c))
        if not term:
            ops.append(qml.Identity(0))
        else:
            pl = [{'X':qml.PauliX,'Y':qml.PauliY,'Z':qml.PauliZ}[p](i) for i,p in term]
            ops.append(pl[0] if len(pl)==1 else qml.operation.Tensor(*pl))
    return qml.Hamiltonian(coeffs, ops)

In [ ]:
# CELL 4: Build molecules and Hamiltonians
# We compare ethylene (alkene) vs acetylene (alkyne) side-by-side

molecules = {
    'ethylene': {
        'geometry': [
            ('C',(0.000, 0.000, 0.000)),('C',(0.000,0.000,1.339)),
            ('H',(0.000,0.926,-0.546)),('H',(0.000,-0.926,-0.546)),
            ('H',(0.000,0.926,1.885)),('H',(0.000,-0.926,1.885)),
        ],
        'freeze_core': [0,1,2],   # freeze 3 sigma core orbitals
        'frozen_e': 6,
        'pi_bonds': 1,
        'color': 'steelblue',
    },
    'acetylene': {
        'geometry': [
            ('C',(0.000,0.000,0.000)),('C',(0.000,0.000,1.203)),
            ('H',(0.000,0.000,-1.063)),('H',(0.000,0.000,2.266)),
        ],
        'freeze_core': [0],        # freeze 1 sigma core
        'frozen_e': 2,
        'pi_bonds': 2,             # TWO orthogonal pi bonds
        'color': 'darkorange',
    },
}

mol_data = {}
hamiltonians = {}

for name, info in molecules.items():
    md = MolecularData(geometry=info['geometry'], basis='sto-3g',
                       multiplicity=1, charge=0, description=name)
    md = run_pyscf(md, run_scf=True, run_ccsd=True, run_fci=True)
    mol_data[name] = md
    
    fham = get_fermion_operator(md.get_molecular_hamiltonian())
    active_fham = freeze_orbitals(fham, occupied=info['freeze_core'], virtual=[])
    jw = jordan_wigner(active_fham)
    nq = count_qubits(jw)
    ne = md.n_electrons - info['frozen_e']
    hamiltonians[name] = {'jw': jw, 'pl': of_to_pl(jw), 'n_qubits': nq, 'n_electrons': ne}
    
    corr = (md.fci_energy - md.hf_energy)*1000
    print(f'{name}: {nq} active qubits, {ne} active e⁻, '          f'FCI={md.fci_energy:.6f} Ha, corr={corr:.2f} mHa, π-bonds={info["pi_bonds"]}')

In [ ]:
# CELL 5: UCCSD-VQE runner (fixed ansatz)
def run_uccsd_vqe(H_pl, n_qubits, n_electrons, max_iter=200, stepsize=0.4):
    singles, doubles = qchem.excitations(n_electrons, n_qubits)
    hf_state = qchem.hf_state(n_electrons, n_qubits)
    dev = qml.device('default.qubit', wires=n_qubits)
    @qml.qnode(dev)
    def circuit(params):
        qml.BasisState(hf_state, wires=range(n_qubits))
        qml.AllSinglesDoubles(params, wires=range(n_qubits),
            hf_state=hf_state, singles=singles, doubles=doubles)
        return qml.expval(H_pl)
    params = np.zeros(len(singles)+len(doubles))
    opt = qml.GradientDescentOptimizer(stepsize=stepsize)
    energies = []
    for step in range(max_iter):
        params, energy = opt.step_and_cost(circuit, params)
        energies.append(float(energy))
        if step>5 and abs(energies[-1]-energies[-2])<1e-9: break
    est_cnots = len(doubles)*8 + len(singles)*2
    return {'energy': energies[-1], 'history': energies,
            'n_params': len(singles)+len(doubles), 'est_cnots': est_cnots}
print('UCCSD-VQE runner defined.')

In [ ]:
# CELL 6: ADAPT-VQE runner (adaptive ansatz)
# Implements Grimsley et al. Nat. Commun. 2019

def run_adapt_vqe(H_pl, n_qubits, n_electrons, grad_thresh=1e-3, max_ops=20, max_iter=300):
    singles, doubles = qchem.excitations(n_electrons, n_qubits)
    hf_state = qchem.hf_state(n_electrons, n_qubits)
    pool = [('S', s) for s in singles] + [('D', d) for d in doubles]
    dev = qml.device('default.qubit', wires=n_qubits)
    
    selected = []   # (type, excitation_wires)
    params = np.array([])
    energy_history = []
    
    print(f'  Pool: {len(pool)} operators | threshold={grad_thresh}')
    
    for adapt_step in range(max_ops):
        # --- gradient screening ---
        grads = []
        for op_type, exc in pool:
            @qml.qnode(dev)
            def probe(theta, exc=exc, op_type=op_type):
                qml.BasisState(hf_state, wires=range(n_qubits))
                for idx2,(t2,e2) in enumerate(selected):
                    if t2=='S': qml.SingleExcitation(params[idx2], wires=e2)
                    else:       qml.DoubleExcitation(params[idx2], wires=e2)
                if op_type=='S': qml.SingleExcitation(theta, wires=exc)
                else:            qml.DoubleExcitation(theta, wires=exc)
                return qml.expval(H_pl)
            g = abs((probe(np.pi/2)-probe(-np.pi/2))/2.0)
            grads.append(g)
        
        max_g = max(grads)
        best  = int(np.argmax(grads))
        print(f'  ADAPT {adapt_step+1:2d}: max|∇|={max_g:.5f} → {pool[best][0]}_{pool[best][1]}')
        if max_g < grad_thresh:
            print(f'  Converged (max gradient {max_g:.2e} < {grad_thresh})'); break
        
        selected.append(pool[best])
        params = np.append(params, 0.0)
        
        # --- inner VQE re-optimization ---
        @qml.qnode(dev)
        def adapt_circuit(p):
            qml.BasisState(hf_state, wires=range(n_qubits))
            for idx,(op_type,exc) in enumerate(selected):
                if op_type=='S': qml.SingleExcitation(p[idx], wires=exc)
                else:            qml.DoubleExcitation(p[idx], wires=exc)
            return qml.expval(H_pl)
        
        res = minimize(adapt_circuit, params, method='L-BFGS-B',
                       jac=qml.grad(adapt_circuit),
                       options={'maxiter': max_iter, 'ftol':1e-12})
        params = res.x
        energy_history.append(float(res.fun))
        print(f'           E = {res.fun:.8f} Ha')
    
    n_s = sum(1 for t,_ in selected if t=='S')
    n_d = sum(1 for t,_ in selected if t=='D')
    return {'energy': energy_history[-1] if energy_history else None,
            'history': energy_history, 'n_operators': len(selected),
            'est_cnots': n_s*2+n_d*8, 'selected': selected}
print('ADAPT-VQE runner defined.')

In [ ]:
# CELL 7: Run both methods on Ethylene
name = 'ethylene'
H    = hamiltonians[name]['pl']
nq   = hamiltonians[name]['n_qubits']
ne   = hamiltonians[name]['n_electrons']
fci  = mol_data[name].fci_energy

print('=== Ethylene (C=C, 1 π bond) ===')
print('--- UCCSD-VQE ---')
uccsd_eth = run_uccsd_vqe(H, nq, ne)
print(f'UCCSD  E={uccsd_eth["energy"]:.8f} Ha | Err={abs(uccsd_eth["energy"]-fci)*1e3:.3f} mHa | CNOTs≈{uccsd_eth["est_cnots"]}')

print('--- ADAPT-VQE ---')
adapt_eth = run_adapt_vqe(H, nq, ne)
print(f'ADAPT  E={adapt_eth["energy"]:.8f} Ha | Err={abs(adapt_eth["energy"]-fci)*1e3:.3f} mHa | CNOTs≈{adapt_eth["est_cnots"]} | Ops={adapt_eth["n_operators"]}')

In [ ]:
# CELL 8: Run both methods on Acetylene
name = 'acetylene'
H    = hamiltonians[name]['pl']
nq   = hamiltonians[name]['n_qubits']
ne   = hamiltonians[name]['n_electrons']
fci  = mol_data[name].fci_energy

print('=== Acetylene (C≡C, 2 π bonds) ===')
print('--- UCCSD-VQE ---')
uccsd_acc = run_uccsd_vqe(H, nq, ne)
print(f'UCCSD  E={uccsd_acc["energy"]:.8f} Ha | Err={abs(uccsd_acc["energy"]-fci)*1e3:.3f} mHa | CNOTs≈{uccsd_acc["est_cnots"]}')

print('--- ADAPT-VQE ---')
adapt_acc = run_adapt_vqe(H, nq, ne)
print(f'ADAPT  E={adapt_acc["energy"]:.8f} Ha | Err={abs(adapt_acc["energy"]-fci)*1e3:.3f} mHa | CNOTs≈{adapt_acc["est_cnots"]} | Ops={adapt_acc["n_operators"]}')

In [ ]:
# CELL 9: Publication-ready comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# --- Panel 1: Energy convergence ---
ax = axes[0]
ax.plot(uccsd_eth['history'], 'b-', lw=2, label='UCCSD-VQE (ethylene)')
ax.plot(np.arange(len(adapt_eth['history'])), adapt_eth['history'], 'b--o', lw=2, label='ADAPT-VQE (ethylene)')
ax.axhline(mol_data['ethylene'].fci_energy, color='b', ls=':', alpha=0.5, label='FCI (ethylene)')
ax.plot(uccsd_acc['history'], color='darkorange', lw=2, label='UCCSD-VQE (acetylene)')
ax.plot(np.arange(len(adapt_acc['history'])), adapt_acc['history'], color='darkorange', ls='--', marker='o', lw=2, label='ADAPT-VQE (acetylene)')
ax.axhline(mol_data['acetylene'].fci_energy, color='darkorange', ls=':', alpha=0.5, label='FCI (acetylene)')
ax.set_xlabel('Iteration', fontsize=11); ax.set_ylabel('Energy (Ha)', fontsize=11)
ax.set_title('Energy Convergence', fontsize=12); ax.legend(fontsize=7); ax.grid(alpha=0.3)

# --- Panel 2: CNOT count comparison ---
ax2 = axes[1]
molecules_plot = ['Ethylene\n(C=C)', 'Acetylene\n(C≡C)']
uccsd_cnots = [uccsd_eth['est_cnots'], uccsd_acc['est_cnots']]
adapt_cnots = [adapt_eth['est_cnots'], adapt_acc['est_cnots']]
x = np.arange(2); w=0.35
ax2.bar(x-w/2, uccsd_cnots, w, label='UCCSD-VQE', color='steelblue')
ax2.bar(x+w/2, adapt_cnots, w, label='ADAPT-VQE', color='darkorange')
ax2.set_xticks(x); ax2.set_xticklabels(molecules_plot)
ax2.set_ylabel('Est. CNOT count', fontsize=11)
ax2.set_title('Circuit Depth (NISQ metric)', fontsize=12)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

# --- Panel 3: Error vs FCI ---
ax3 = axes[2]
errors = {
    'Ethylene UCCSD': abs(uccsd_eth['energy']-mol_data['ethylene'].fci_energy)*1000,
    'Ethylene ADAPT': abs(adapt_eth['energy']-mol_data['ethylene'].fci_energy)*1000,
    'Acetylene UCCSD': abs(uccsd_acc['energy']-mol_data['acetylene'].fci_energy)*1000,
    'Acetylene ADAPT': abs(adapt_acc['energy']-mol_data['acetylene'].fci_energy)*1000,
}
colors = ['steelblue','steelblue','darkorange','darkorange']
hatches = ['','/','','//']
bars = ax3.bar(list(errors.keys()), list(errors.values()), color=colors)
for bar, hatch in zip(bars, hatches): bar.set_hatch(hatch)
ax3.axhline(1.6, color='red', ls='--', label='Chemical accuracy (1.6 mHa)')
ax3.set_ylabel('|E_VQE - E_FCI| (mHa)', fontsize=11)
ax3.set_title('Error vs FCI Reference', fontsize=12)
ax3.set_xticklabels(list(errors.keys()), rotation=20, ha='right', fontsize=9)
ax3.legend(); ax3.grid(axis='y', alpha=0.3)

plt.suptitle('ADAPT-VQE vs UCCSD-VQE: Alkene & Alkyne Benchmark (STO-3G)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('adapt_vs_uccsd_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as adapt_vs_uccsd_benchmark.png')

In [ ]:
# CELL 10: Print publication-ready benchmark table
import pandas as pd

table_data = {
    'Molecule':    ['Ethylene (C=C)', 'Ethylene (C=C)', 'Acetylene (C≡C)', 'Acetylene (C≡C)'],
    'Method':      ['UCCSD-VQE', 'ADAPT-VQE', 'UCCSD-VQE', 'ADAPT-VQE'],
    'Energy (Ha)': [uccsd_eth['energy'], adapt_eth['energy'],
                    uccsd_acc['energy'], adapt_acc['energy']],
    'ΔE_FCI (mHa)':[abs(uccsd_eth['energy']-mol_data['ethylene'].fci_energy)*1000,
                    abs(adapt_eth['energy']-mol_data['ethylene'].fci_energy)*1000,
                    abs(uccsd_acc['energy']-mol_data['acetylene'].fci_energy)*1000,
                    abs(adapt_acc['energy']-mol_data['acetylene'].fci_energy)*1000],
    '# Params':    [uccsd_eth['n_params'], adapt_eth['n_operators'],
                    uccsd_acc['n_params'], adapt_acc['n_operators']],
    'Est. CNOTs':  [uccsd_eth['est_cnots'], adapt_eth['est_cnots'],
                    uccsd_acc['est_cnots'], adapt_acc['est_cnots']],
}
df = pd.DataFrame(table_data)
df['Energy (Ha)'] = df['Energy (Ha)'].round(8)
df['ΔE_FCI (mHa)'] = df['ΔE_FCI (mHa)'].round(4)
print(df.to_string(index=False))
df.to_csv('benchmark_table.csv', index=False)
print('\nSaved to benchmark_table.csv')

## Hardware Considerations for Running on IBM Quantum

To run these circuits on real IBM Quantum hardware (notebook 04):

1. **Qubit count:** Active-space selection keeps circuits at 8–12 qubits
   — well within 127-qubit Eagle processor capacity.
2. **ADAPT advantage:** Selecting only 3–8 operators vs 20+ in UCCSD means
   shallower circuits that survive decoherence (T₂ ~ 100 μs on Eagle).
3. **Error mitigation:** Use `ZNE` (Zero Noise Extrapolation) from
   `qiskit_ibm_runtime.options` to push accuracy toward chemical accuracy (1.6 mHa).
4. **Transpilation:** Use `optimization_level=3` in Qiskit for heavy-hex
   topology — BK mapping produces more local operators that map better.
5. **Shots:** 8192–16384 shots recommended for Hamiltonian estimation;
   use Pauli grouping to reduce circuit executions by ~5×.